# GameTheory-16c : La dimension paiement que le designer n'a jamais cherchée — extraction de revenu sous non-déviation locale+IR au premier meilleur

**Objectifs** : énumérer les paiements, comparer les frontières et expliciter les contraintes du jouet.

**Prérequis** : mécanismes binaires, utilité quasi-linéaire ; **durée** : 75 minutes.

**Navigation** : [Version Python](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) | [<< 16b-Automated-Mechanism-Design](GameTheory-16b-Automated-Mechanism-Design-Python.ipynb) | [Index GameTheory](README.md) | [16k-Encheres-Myerson >>](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb)

GT-16b a posé le triplet de l'automated mechanism design sur un domaine jouet : un **générateur** qui énumère et choisit, un **vérificateur indépendant** qui contrôle non-déviation locale et IR, un **témoin d'impossibilité**. Mais son générateur a un aveugle : il n'énumère que les **issues** — les paiements sont codés en dur à zéro. Et son témoin d'impossibilité, qui condamne les paiements strictement positifs, prouve son énoncé via un agent de **type 0**… alors que le vérificateur du notebook checke l'IR sur les **types vrais** de l'instance — sur l'instance où les deux agents sont de type 1, l'argument du témoin ne tire jamais.

Cette variante -c remet la dimension paiement dans l'énumération et mesure les deux écarts. Premier écart, la table : sur l'instance (1,1), **47 mécanismes non-déviation locale+IR existent, dont 29 à bien-être optimal avec revenu strictement positif — le designer committé en atteint zéro**. La forme close du revenu maximal est lisible : `Σθᵢ`. Deuxième écart, la couture de lectures : sous l'IR **uniforme sur les types et reports** que la preuve du témoin présuppose (tout type pourrait se présenter), les paiements ≡ 0 ne sont pas un choix mais un **théorème** — sous l'IR **point-prior** que le vérificateur implémente, un mécanisme à paiement uniforme survit sur (1,1) et extrait tout le surplus. Les 29 mécanismes à revenu positif vivent uniquement d'un côté de la couture.

Références : GT-16b (`GameTheory-16b-Automated-Mechanism-Design-Python.ipynb`) · Conitzer & Sandholm (2002), *Automated Mechanism Design* · Crémer & McLean (1988), parenté de l'extraction à prior corrélé.

**Portée des vérificateurs.** « non-déviation locale local » désigne ici le test hérité qui fixe les deux types et compare tous les reports unilatéraux depuis chaque profil. Ce n’est pas la définition usuelle de non-déviation locale pour chaque type au report sincère. L’IR dite uniforme contrôle chaque type à chaque report, sans espérance : elle ne doit pas être confondue avec l’IR ex-interim usuelle. Les tableaux décrivent ces contraintes explicites du jouet ; aucune application du théorème de Crémer–McLean n’est revendiquée.

Dans la terminologie habituelle, DSIC concerne les incitations au report sincère pour chaque type. Ici les résultats revendiqués portent seulement sur la non-déviation locale explicitement définie et mesurée. Le générateur hérité est réellement soumis à ce test : son échec n’est pas assimilé à une admissibilité.

## Objectifs d'apprentissage

1. **Lire un espace de recherche** : repérer que le générateur de GT-16b n'énumère que les issues et que ses paiements ≡ 0 sont un choix de design — designer dans un sous-espace et le nommer.
2. **Énumérer l'espace complet** : 2⁴ issues × tables de paiement {0,1} — 4096 candidats — filtrés par le même vérificateur indépendant (copie locale, zéro import), et mesurer la frontière revenu / bien-être sur les 4 instances point-prior.
3. **Exhiber le mécanisme extrateur** et le faire vérifier cellule par cellule — non-déviation locale, IR, bien-être, revenu — par l'organe hérité de GT-16b.
4. **Lire une preuve d'impossibilité dans sa sémantique** : le témoin de GT-16b est vrai — et même renforcé — en lecture uniforme sur les types et reports, faux sur (1,1) en lecture point-prior ; localiser la couture entre les deux et la mesurer des deux côtés.
5. **Écrire les bornes honnêtes** : forme close du revenu (`Σθᵢ`), lien Crémer-McLean (extraction complète = propriété du prior ponctuel), ce que le jouet ne dit pas hors prior ponctuel.

## 1. Ce que le designer committé designe vraiment

Le domaine de GT-16b : 2 agents, types θ ∈ {0,1}, issue o ∈ {0,1}, paiements ∈ {0,1}². Un mécanisme est un couple de tables : `issue(r) ∈ {0,1}` et `paiement(r) ∈ {0,1}²` pour chacun des 4 profils de reports `r`. L'espace COMPLET des mécanismes compte `2⁴ × 4⁴ = 4096` candidats. Le générateur de GT-16b n'en visite que `2⁴ = 16` : il énumère les issues, prend l'argmax de bien-être, puis **fixe tous les paiements à zéro** (« les paiements minimaux qui satisfont trivialement IR », dit son commentaire). Le choix est défendable — mais c'est un choix : l'utilité quasi-linéaire `uᵢ = θᵢ·issue − paiementᵢ` rend les transferts neutres pour le bien-être `J = Σθᵢ·issue`, donc ignorer les paiements ne coûte rien au critère optimisé… et coûte tout au seul autre critère naturel, le revenu.

Reproduisons d'abord le designer committé sur l'instance `true = (1,1)` — le point-prior où les deux agents sont de type 1.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");
record Mechanisme(int[] Issues,int[,] Paiements);
int Type(int profil,int agent)=>(profil>>(1-agent))&1;
int[] Issues(int masque)=>Enumerable.Range(0,4).Select(r=>(masque>>(3-r))&1).ToArray();
IEnumerable<Mechanisme> EnumererTous(int maximum=1,int minimum=0){
 int radix=maximum-minimum+1; int count=(int)Math.Pow(radix,8);
 for(int mask=0;mask<16;mask++)for(int number=0;number<count;number++){var pay=new int[4,2];int q=number;for(int k=7;k>=0;k--){pay[k/2,k%2]=minimum+q%radix;q/=radix;}yield return new Mechanisme(Issues(mask),pay);}
}
(Mechanisme mecanisme,int bienEtre) GenererGT16b(int type0,int type1){int best=-1,profil=2*type0+type1;int[] choix=null;for(int mask=0;mask<16;mask++){int o=Issues(mask)[profil],j=(type0+type1)*o;if(j>best){best=j;choix=Issues(mask);}}return(new Mechanisme(choix,new int[4,2]),best);}
var generateur=GenererGT16b(1,1);
Console.WriteLine($"Instance point-prior : true=(1,1), issue={generateur.mecanisme.Issues[3]}, bien-être={generateur.bienEtre}, revenu=0");
Console.WriteLine("Générateur : 16 tables d'issues ; espace complet : 16 × 256 = 4096 mécanismes");

Runtime : .NET 10.0.14


Instance point-prior : true=(1,1), issue=1, bien-être=2, revenu=0


Générateur : 16 tables d'issues ; espace complet : 16 × 256 = 4096 mécanismes


**Lecture de la sortie committée** : le designer atteint le premier meilleur (`J = 2` : l'issue vaut 1 au profil vrai, chaque agent de type 1 en tire 1) avec un revenu nul. Les deux nombres de la dernière ligne disent tout : le générateur visite 16 candidats sur 4096 — **0,4 % de l'espace des mécanismes**. Rien dans le domaine n'impose ce choix : IR se satisfait aussi avec des paiements positifs, et non-déviation locale se vérifie sur n'importe quelle table. La question de la section 2 est donc : que contiennent les 99,6 % restants ?

## 2. L'énumération conjointe — le vérificateur hérité tranche

On hérite l'organe de GT-16b : son **vérificateur indépendant**, en copie locale, zéro import du générateur — mêmes définitions de l'utilité quasi-linéaire, de non-déviation locale (aucune déviation unilatérale strictement profitable, pour chaque agent et chaque profil reporté) et d'IR (utilité ≥ 0 pour chaque agent et chaque profil, **aux types vrais** — la sémantique exacte de `verify_IR` de GT-16b). Puis on énumère les 4096 candidats de l'espace complet, et on compte par instance point-prior : les admissibles (non-déviation locale ∧ IR), parmi eux les à bien-être optimal, parmi ceux-ci les à revenu strictement positif, et le revenu maximal.

In [2]:
bool VerifierDsicLocal(Mechanisme m,int type0,int type1){for(int r=0;r<4;r++)for(int i=0;i<2;i++){int theta=i==0?type0:type1;int autre=r^(1<<(1-i));int sincere=theta*m.Issues[r]-m.Paiements[r,i],dev=theta*m.Issues[autre]-m.Paiements[autre,i];if(dev>sincere)return false;}return true;}
bool VerifierIR(Mechanisme m,int type0,int type1){for(int r=0;r<4;r++)for(int i=0;i<2;i++)if((i==0?type0:type1)*m.Issues[r]-m.Paiements[r,i]<0)return false;return true;}
int BienEtre(Mechanisme m,int type0,int type1)=>(type0+type1)*m.Issues[2*type0+type1];
int Revenu(Mechanisme m,int type0,int type1)=>m.Paiements[2*type0+type1,0]+m.Paiements[2*type0+type1,1];
var frontiere=new Dictionary<int,(int admissibles,int premierMeilleur,int revenuPositif,int revenuMax)>();
Console.WriteLine("Types | admissibles | premier meilleur | revenu positif | revenu maximal");
for(int profil=0;profil<4;profil++){int t0=Type(profil,0),t1=Type(profil,1);int admissibles=0,fb=0,pos=0,best=-1;foreach(var m in EnumererTous()){if(VerifierDsicLocal(m,t0,t1)&&VerifierIR(m,t0,t1)){admissibles++;if(BienEtre(m,t0,t1)==t0+t1){fb++;int rev=Revenu(m,t0,t1);if(rev>0)pos++;best=Math.Max(best,rev);}}}frontiere[profil]=(admissibles,fb,pos,best);Console.WriteLine($"({t0},{t1}) | {admissibles} | {fb} | {pos} | {best}");}
Console.WriteLine("Vérificateur local : aucune déviation profitable aux types fixés, IR sur tous les reports");
Console.WriteLine($"Générateur hérité : non-déviation locale={VerifierDsicLocal(generateur.mecanisme,1,1)}, IR={VerifierIR(generateur.mecanisme,1,1)} ; agent 0 : report (0,1), utilité=0, déviation (1,1), utilité=1");

Types | admissibles | premier meilleur | revenu positif | revenu maximal


(0,0) | 16 | 16 | 0 | 0


(0,1) | 25 | 15 | 10 | 1


(1,0) | 25 | 15 | 10 | 1


(1,1) | 47 | 34 | 29 | 2


Vérificateur local : aucune déviation profitable aux types fixés, IR sur tous les reports


Générateur hérité : non-déviation locale=False, IR=True ; agent 0 : report (0,1), utilité=0, déviation (1,1), utilité=1


**Lecture de la sortie committée** : la colonne qui condamne l'effondrement est la dernière. Sur `(1,1)` : **47 mécanismes non-déviation locale+IR**, dont **34 à bien-être optimal** (`issue((1,1)) = 1`), parmi lesquels **29 à revenu strictement positif**, pour un revenu maximal de **2**. Le designer committé (paiements ≡ 0) atteint le bien-être maximal, mais sa table ne satisfait pas la non-déviation locale ; il laisse 2/2 = **100 % du revenu atteignable sur la table**. La ligne `(0,0)` est le contre-point structurel : tous les types nuls ⇒ 16 admissibles, tous à bien-être « optimal » (fb = 0), aucun à revenu positif — IR force `paiementᵢ = 0` pour tout agent de type 0 (son utilité est `−paiementᵢ`, indépendante de l'issue). Et la forme close se lit dans la dernière colonne : **max revenu @ premier meilleur = Σθᵢ** — 2, 1, 1, 0.

## Exercice 1 : l'instance (0,0) est une forteresse

La table de la section 2 montre 16 mécanismes admissibles sur `(0,0)` et aucun à revenu positif. Démontrez-le proprement : pour chaque agent de type 0, écrivez la contrainte IR point-prior et montrez qu'elle force `paiementᵢ(r) = 0` pour tout profil `r` — indépendamment de l'issue. Concluez : le revenu maximal d'une instance est nul dès que tous les types sont nuls, et la forme close `Σθᵢ` est cohérente. Vérifiez enfin votre preuve par énumération sur les deux autres instances à agent de type 0 : leur revenu ne peut venir que de l'agent intéressé.

Complétez `borne_revenu` (Python) ou `borneRevenu` (C#), pour deux types binaires et une issue binaire ; retournez la borne totale de revenu impliquée par IR. Le garde mélange cas nuls et non nuls, puis votre preuve explique pourquoi un type nul ne finance rien.

In [3]:
// Exercice 1 a completer
// 1. Pour true = (0,0) : ecrire la contrainte IR d'un agent de type 0 (u_i = 0*issue - p_i)
// 2. En deduire p_i(r) = 0 pour tout r, puis revenu = 0 pour TOUT mecanisme admissible
// 3. Verifier : FRONTIERE[(0,0)][2] == 0 et FRONTIERE[(0,0)][3] == 0 ; puis (0,1) : revenu <= 1
// Indice : u_i = -p_i(r) >= 0 et p_i(r) dans {0,1} => p_i(r) = 0
Func<int,int,int,double?> borneRevenu=(type0,type1,issue)=>{
 // TODO : borne issue de l'IR pour les types fixés et l'issue donnée
 return null;
};
var casBorne=new[]{(0,0,0,0.0),(0,0,1,0.0),(0,1,1,1.0),(1,0,1,1.0),(1,1,1,2.0),(1,1,0,0.0)};
var reponsesBorne=casBorne.Select(c=>borneRevenu(c.Item1,c.Item2,c.Item3)).ToArray();
if(reponsesBorne.All(x=>x is null))Console.WriteLine("Exercice 1 à compléter : borne IR et preuve pour les types nuls");
else {for(int k=0;k<casBorne.Length;k++)if(reponsesBorne[k] is null||!double.IsFinite(reponsesBorne[k].Value)||Math.Abs(reponsesBorne[k].Value-casBorne[k].Item4)>1e-9)throw new Exception("Borne incorrecte");Console.WriteLine("Exercice 1 : bornes validées ; rédigez la preuve et relisez la frontière (0,0), (0,1), (1,0)");}

Exercice 1 à compléter : borne IR et preuve pour les types nuls


## 3. Le mécanisme extrateur — exhibé et vérifié par l'organe hérité

L'argmax revenu au premier meilleur sur `(1,1)` est lisible à l'œil dans l'énumération : **issue = 1 uniquement au profil (1,1)** (le bien-être du point-prior ne regarde que le profil vrai), **chaque agent paie 1 au profil (1,1)** — sa pleine valeur — et **tout le reste à zéro**. La logique : IR est serré à l'indifférence pour les types vrais (`uᵢ = 1·1 − 1 = 0`), et non-déviation locale tient par indifférence faible — toute déviation mène à un profil où l'issue est 0 et le paiement 0, d'utilité 0, jamais strictement mieux que 0. C'est la discrimination parfaite du point-prior : le designer connaît les types, il facture exactement la valeur.

In [4]:
var extracteur=new Mechanisme(new[]{0,0,0,1},new int[,]{{0,0},{0,0},{0,0},{1,1}});
Console.WriteLine($"Extracteur : non-déviation locale local={VerifierDsicLocal(extracteur,1,1)}, IR={VerifierIR(extracteur,1,1)}, bien-être={BienEtre(extracteur,1,1)}, revenu={Revenu(extracteur,1,1)}");
int sincere=extracteur.Issues[3]-extracteur.Paiements[3,0],deviation=extracteur.Issues[1]-extracteur.Paiements[1,0];
Console.WriteLine($"Agent 1 : utilité sincère={sincere}, après déviation={deviation} ; indifférence faible");
Console.WriteLine($"Contre-témoin : même bien-être {generateur.bienEtre}, revenu {Revenu(extracteur,1,1)} contre 0 pour le générateur");

Extracteur : non-déviation locale local=True, IR=True, bien-être=2, revenu=2


Agent 1 : utilité sincère=0, après déviation=0 ; indifférence faible


Contre-témoin : même bien-être 2, revenu 2 contre 0 pour le générateur


**Lecture de la sortie committée** : le mécanisme extrateur passe les quatre contrôles de l'organe hérité — non-déviation locale, IR, bien-être au premier meilleur, revenu maximal. Le détail de la déviation montre la structure : l'agent sincère est à utilité 0 (IR serré), la déviation lui donne 0 — l'indifférence faible suffit à non-déviation locale, qui n'exige que l'absence de déviation **strictement** profitable. La lecture économique est honnête et bornée : c'est **l'extraction complète à prior ponctuel** — le designer connaît les types vrais et designe pour cette seule instance, un cas d’information complète pour le designer ; aucun théorème de Crémer–McLean n’est appliqué ici. Hors prior ponctuel, avec des types distribués, le problème devient la vraie AMD de Conitzer-Sandholm et cette garantie disparaît : question ouverte, pas une claim du jouet.

## Exercice 2 : élargir la gamme des paiements

Toute la -c utilise des paiements ∈ {0, 1}. Sur l'instance `(1,1)`, élargissez à {0, 1, 2} et relancez l'énumération conjointe (l'espace croît à `2⁴ × 9⁴` — reste trivial). Écrivez d'abord la borne : IR point-prior pour un agent de type 1 exige `paiementᵢ(r) ≤ issue(r) ≤ 1` — le revenu au profil vrai est donc plafonné à 2, quelle que soit la gamme. Confirmez par l'énumération, puis observez ce que la gamme étendue ne change PAS : le bien-être `J` ne compte pas les paiements — dites ce que ça révèle de la définition du bien-être du jouet (un transfert de l'agent au designer y est invisible : ni coût, ni benefice social).

Complétez `mesurer_frontiere` / `mesurerFrontiere` : consommez l’itérable des mécanismes fourni et retournez le quadruplet (admissibles, premier meilleur, revenu positif, revenu maximal). Le garde observe le nombre de candidats réellement parcourus, dont les 104 976 candidats de la gamme étendue.

In [5]:
// Exercice 2 a completer
// 1. Prediction ecrite AVANT : max revenu @ fb sur (1,1) avec paiements dans {0,1,2} = ?
// 2. Enumerer (issues 2^4) x (paiements {0,1,2}^(2*4)) avec le meme verificateur
// 3. Comparer au max = 2 : la borne vient de IR (p_i <= issue <= 1), pas de la gamme
// 4. Bonus : J change-t-il avec les paiements ? (J ne compte que les utilites grossieres)
// Indice : pour l'agent i de type 1, IR en r exige 1*issue(r) - p_i(r) >= 0, et issue(r) <= 1
Func<IEnumerable<Mechanisme>,int,int,(int,int,int,int)?> mesurerFrontiere=(candidats,type0,type1)=>{
 // TODO : consommer l'énumération et compter admissibles, premier meilleur, revenu positif et revenu maximal
 return null;
};
var casFrontiere=new[]{(1,0,0),(1,0,1),(1,1,0),(1,1,1),(2,1,1),(2,0,1)};
int completes=0;
foreach(var (maxPaiement,t0,t1) in casFrontiere){int visites=0;IEnumerable<Mechanisme> Observer(){foreach(var m in EnumererTous(maxPaiement)){visites++;yield return m;}}var rep=mesurerFrontiere(Observer(),t0,t1);if(rep is null)continue;completes++;var f=frontiere[2*t0+t1];if(rep.Value!=(f.admissibles,f.premierMeilleur,f.revenuPositif,f.revenuMax)||visites!=16*(int)Math.Pow(maxPaiement+1,8))throw new Exception("Frontière ou budget incorrect");}
if(completes==0)Console.WriteLine("Exercice 2 à compléter : gamme de paiements étendue");else if(completes!=casFrontiere.Length)throw new Exception("Réponse partielle");else Console.WriteLine("Exercice 2 : six frontières et budgets d'énumération validés");

Exercice 2 à compléter : gamme de paiements étendue


## 4. La couture du témoin d'impossibilité — deux lectures d'IR qui ne se rencontrent pas

Le témoin de GT-16b démontre : `non-déviation locale ∧ IR ∧ (∀r, ∀i : paiementᵢ(r) ≥ 1)` ⇒ ensemble vide. Sa preuve : pour un agent de **type 0**, `uᵢ = −paiementᵢ ≤ −1 < 0`, IR violée. Relisez le vérificateur de la section 2 : `VerifierIR` évalue l'utilité **aux types vrais de l'instance**. L'argument du témoin ne tire que si un agent de type 0 **existe** dans l'instance. Sur `(1,1)` il n'existe pas — la preuve est vide, et la question devient empirique : existe-t-il un mécanisme à paiement uniforme ≥ 1 admissible sur (1,1) ?

Les deux lectures d'IR méritent d'être nommées, car le jouet les mélange sans le dire :

- **IR point-prior** (ce que `verify_IR` de GT-16b implémente, et donc tout son notebook exécute) : `uᵢ ≥ 0` pour les types **vrais** — le designer sait qui est là.
- **IR uniforme sur les types et reports** (ce que la preuve du témoin présuppose : « tout type pourrait se présenter ») : `uᵢ ≥ 0` pour **chaque type possible** de chaque agent — le designer ne sait pas qui va se présenter.

On mesure les trois côtés de la couture : le témoin reproduit là où son agent de type 0 existe, son contre-témoin sur (1,1), et la ligne d'arrivée uniforme sur les types et reports complète.

La seconde contrainte est volontairement robuste : le paiement doit rester acceptable même lorsque type et report diffèrent. L’IR ex-interim standard utilise une espérance conditionnelle sur les types des autres agents et n’est pas calculée ici.

In [6]:
Console.WriteLine("Paiements ≥1 partout : IR point-prior, instance par instance");
for(int profil=0;profil<4;profil++){int t0=Type(profil,0),t1=Type(profil,1);int count=EnumererTous(2,1).Count(m=>VerifierDsicLocal(m,t0,t1)&&VerifierIR(m,t0,t1));Console.WriteLine($"({t0},{t1}) : {count} admissible(s)");}
var survivant=new Mechanisme(Enumerable.Repeat(1,4).ToArray(),new int[,]{{1,1},{1,1},{1,1},{1,1}});
Console.WriteLine($"Survivant uniforme (1,1) : non-déviation locale local={VerifierDsicLocal(survivant,1,1)}, IR={VerifierIR(survivant,1,1)}, bien-être={BienEtre(survivant,1,1)}, revenu={Revenu(survivant,1,1)}");
bool VerifierIRUniforme(Mechanisme m){for(int r=0;r<4;r++)for(int i=0;i<2;i++)for(int theta=0;theta<2;theta++)if(theta*m.Issues[r]-m.Paiements[r,i]<0)return false;return true;}
var robustes=EnumererTous().Where(m=>VerifierIRUniforme(m)&&VerifierDsicLocal(m,1,1)).ToArray();
int nonNuls=robustes.Count(m=>Enumerable.Range(0,8).Any(k=>m.Paiements[k/2,k%2]!=0));
Console.WriteLine($"IR uniforme sur types et reports : {robustes.Length} admissibles, {nonNuls} avec paiement non nul ; revenus=[{string.Join(",",robustes.Select(m=>Revenu(m,1,1)).Distinct().Order())}]");
Console.WriteLine("La contrainte robuste inclut theta=0 à chaque report : tout paiement est nul. Ce n'est pas l'IR ex-interim usuelle.");

Paiements ≥1 partout : IR point-prior, instance par instance


(0,0) : 0 admissible(s)


(0,1) : 0 admissible(s)


(1,0) : 0 admissible(s)


(1,1) : 1 admissible(s)


Survivant uniforme (1,1) : non-déviation locale local=True, IR=True, bien-être=2, revenu=2


IR uniforme sur types et reports : 2 admissibles, 0 avec paiement non nul ; revenus=[0]


La contrainte robuste inclut theta=0 à chaque report : tout paiement est nul. Ce n'est pas l'IR ex-interim usuelle.


**Lecture de la sortie committée** : les trois côtés coexistent sans contradiction — mais ils requalifient l'énoncé de GT-16b. **(1)** Le témoin est vrai sur les trois instances qui contiennent un agent de type 0 : la contrainte uniforme y produit un ensemble vide, sa preuve tire exactement là où elle le prétend. **(2)** Sur (1,1), le contre-témoin existe : `issue ≡ 1, paiements ≡ (1,1)` passe non-déviation locale et IR point-prior et extrait le revenu maximal 2 — un mécanisme à paiement uniforme **et** à extraction complète. La conclusion imprimée par GT-16b (« l'ensemble des mécanismes admissibles est VIDE ») est, en sémantique point-prior, une sur-généralisation : vraie instance par instance, sauf sur celle qui intéresse le plus le vendeur. **(3)** En lecture uniforme sur les types et reports, tout s'inverse : θ = 0 force `pᵢ(r) = 0` pour tout r — **aucun** paiement positif n'est possible, l'effondrement de paiements ≡ 0 n'était pas un choix de design mais un théorème, et le témoin est vrai et renforcé (il condamnait l'uniforme ; la lecture uniforme sur les types et reports condamne tout paiement). La couture est le résultat central de la -c : la preuve du témoin raisonne uniforme sur les types et reports, le vérificateur checke point-prior, et les 29 mécanismes à revenu positif vivent uniquement du côté où la preuve ne s'applique pas.

## Exercice 3 : non-déviation locale strict

L'extrateur satisfait non-déviation locale par indifférence faible (u sincère = u déviation = 0). Construisez la variante stricte : un mécanisme est non-déviation locale-strict si toute déviation est **strictement** moins bien (u sincère > u déviation). Attention au domaine : utilités et paiements entiers ⇒ les écarts sont entiers — réfléchissez à ce qu'un « strict » peut discriminer ici. Parmi les 29 mécanismes à revenu positif de `(1,1)`, combien survivent au non-déviation locale strict ? Le revenu maximal change-t-il ? Et le survivant uniforme de la section 4 (issue ≡ 1, paiements ≡ (1,1)) — que devient-il ? Concluez sur la robustesse de l'extraction : faut-il l'indifférence pour extraire, ou peut-on extraire en interdisant même les déviations neutres ?

Complétez d’abord `marges_locales` / `margesLocales` : retournez huit différences « utilité courante moins utilité après changement de report », en parcourant les quatre profils puis les deux agents. Une marge strictement positive accepte localement la stricte non-déviation. Le garde inclut des marges négatives et positives ; il en déduit ensuite le nombre de mécanismes stricts. Pensez au fait que les deux sens d’un même changement figurent dans le vérificateur local.

In [7]:
// Exercice 3 a completer
// 1. Definir verif_dsic_strict(M, true) : u_sincere > u_dev pour toute deviation
// 2. Re-enumerer l'espace complet sur (1,1) : compter les non-déviation locale-strict + IR a welfare optimal
// 3. Max revenu sous non-déviation locale strict vs 2 (faible) ; et le survivant uniforme de la section 4 ?
// Indice : sur ce domaine, les ecarts d'utilite sont entiers -- l'egalite exacte est le seul cas filtre
Func<Mechanisme,int,int,double[]?> margesLocales=(m,type0,type1)=>{
 // TODO : huit marges utilité courante moins utilité après changement de report ; ordre profil puis agent
 return null;
};
var sansPaiement=new Mechanisme(new[]{0,0,0,1},new int[4,2]);
var casMarges=new[]{(extracteur,new double[8]),(survivant,new double[8]),(sansPaiement,new[]{0.0,0.0,-1.0,0.0,0.0,-1.0,1.0,1.0})};
int reponsesMarges=0;foreach(var (m,attendu) in casMarges){var rep=margesLocales(m,1,1);if(rep is null)continue;reponsesMarges++;if(rep.Length!=8||rep.Any(x=>!double.IsFinite(x))||rep.Where((x,k)=>Math.Abs(x-attendu[k])>1e-9).Any())throw new Exception("Marges incorrectes");}
if(reponsesMarges==0)Console.WriteLine("Exercice 3 à compléter : non-déviation stricte locale");else if(reponsesMarges!=casMarges.Length)throw new Exception("Réponse partielle");else {int stricts=EnumererTous().Where(m=>VerifierDsicLocal(m,1,1)&&VerifierIR(m,1,1)&&BienEtre(m,1,1)==2&&Revenu(m,1,1)>0).Count(m=>{var ms=margesLocales(m,1,1);if(ms is null||ms.Length!=8||ms.Any(x=>!double.IsFinite(x)))throw new Exception("Marges incomplètes");if(ms.Any(x=>Math.Abs(x)>1e-9))throw new Exception("Marges incompatibles avec le vérificateur local : les deux sens imposent une égalité");return ms.All(x=>x>0);});Console.WriteLine($"Exercice 3 : marges validées, {stricts} mécanismes à revenu positif satisfont toutes les inégalités strictes locales");}

Exercice 3 à compléter : non-déviation stricte locale



(6,33): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



## 5. Verdict — ce que l'effondrement coûte, ce qui survit

| Question | Réponse mesurée | Statut |
|---|---|---|
| Le designer GT-16b était-il admissible ? | Non : IR et bien-être maximal, mais non-déviation locale violée par la table choisie | **cède** — le vérificateur révèle le défaut |
| Son bien-être était-il optimal ? | Oui — les transferts sont neutres pour `J = Σθᵢ·issue` | **survit** |
| Son revenu ? | 0, contre un maximum de `Σθᵢ` — **100 % du revenu atteignable laissé sur la table** sur chaque instance à types 1 | **cède** (le critère ignoré) |
| Le témoin d'impossibilité ? | Vrai là où un agent de type 0 existe (3 instances sur 4) ; contredit sur (1,1) en point-prior (1 survivant uniforme, extraction complète) ; vrai et renforcé en uniforme sur les types et reports (tout paiement positif impossible) | **survit borné** — portée réécrite, couture localisée |
| Forme close du revenu maximal @ premier meilleur ? | `Σθᵢ` — les 4 instances la vérifient (2, 1, 1, 0) | **acquis** |

Le bilan de maturation est le même que pour GT-18b : le jouet **survit borné**. La vérification explicite rejette la non-déviation locale du générateur, sans invalider son bien-être maximal ni son revenu nul. Le générateur designait dans 0,4 % de l'espace des mécanismes, et le témoin bornait une contrainte que sa propre sémantique d'exécution n'impose pas sur l'instance la plus favorable. La -c ajoute la dimension manquante : la frontière revenu/bien-être complète, le mécanisme extrateur vérifié par l'organe hérité, et la localisation exacte de la couture point-prior / uniforme sur les types et reports.

### Questions ouvertes — ce que ce jouet ne dit pas

- **Hors prior ponctuel.** L'extraction complète tient parce que le designer connaît les types vrais avec certitude. Avec un prior distribué (même indépendant, même à 2 types), la contrainte non-déviation locale devient liante et le revenu maximal exige la vraie machinerie AMD (énumération sur les distributions, pas sur les instances). La forme close `Σθᵢ` est une propriété du point-prior, pas du problème.
- **Le budget.** Le jouet n'a pas de coût d'issue (proviser l'issue 1 est gratuite). Avec un coût `c`, la question devient : le revenu extrayable finance-t-il la provision ? — la vraie structure du problème du bien public, et le théorème de Green-Laffont entre par cette porte.
- **L'indifférence faible.** non-déviation locale est satisfait par indifférence (u = 0 partout pour les types vrais de l'extrateur). En pratique, un agent exactement indifférent peut dévier pour un dixième de raison ; la variante stricte sur tous les reports impose deux écarts opposés strictement positifs et est donc impossible ; les marges signées se construisent dans l’exercice 3 — mesurable dans le même jouet, non mesurée ici (exercice 3).

## Exercices — récapitulatif

Les trois activités réparties démontrent la borne IR des types nuls, mesurent la gamme de paiements étendue en conservant son budget, puis construisent les marges de non-déviation stricte. Rédigez les arguments qui relient vos sorties aux limites du modèle.

## À retenir

GT-16b avait construit l'organe juste — un vérificateur indépendant qui tranche sans importer le générateur — et l'avait arrêté à mi-chemin de son propre espace. Quatre acquis :

1. **La mesure de l'effondrement** : le générateur visitait 16 candidats sur 4096 ; en réparant, 29 mécanismes à bien-être optimal ET à revenu positif existent sur `(1,1)` — l'ignorance des paiements coûtait 100 % du revenu, pas une marge.
2. **La forme close** : revenu maximal au premier meilleur = `Σθᵢ`, atteint par le mécanisme extrateur — pleine valeur facturée aux agents intéressés au profil vrai, IR serré, non-déviation locale par indifférence. Une instance, pas un théorème général : l'extraction complète est la propriété du prior ponctuel.
3. **La couture localisée** : la preuve du témoin raisonne uniforme sur les types et reports, le vérificateur checke point-prior. Sur les instances à agent de type 0 les lectures s'accordent (témoin vrai) ; sur (1,1) elles divergent — 29 mécanismes à revenu positif d'un côté, `paiements ≡ 0` en théorème de l'autre. Une impossibilité sans contre-témoin surnomme sa contrainte ; avec contre-témoin, elle en montre la bordure.
4. **La méthode** : le même geste s'applique à chaque designer borné — énumérer l'espace qu'il visite, l'espace complet, mesurer l'écart sur **tous** les critères du problème (pas seulement celui qu'il optimise), et nommer la sémantique exacte dans laquelle chaque preuve raisonne.

Perspectives immédiates dans le même jouet : le coût d'issue nul (le bien public de Green-Laffont), le non-déviation locale strict de l'exercice 3, et la sortie du prior ponctuel — la vraie AMD où ces 4096 candidats deviennent des familles indexées par les distributions.
**Trois mesures à retenir.**

- Le générateur de `GameTheory-16b` visite **16** tables d'issues ; l'espace complet en compte `16 × 256 = 4096`. Sur le profil `(1, 1)`, il atteint `J = 2` (le premier meilleur) pour un **revenu nul**.
- Le même organe, appliqué à l'espace complet, trouve sur `(1, 1)` **47** mécanismes admissibles, dont **34** à bien-être optimal et **29** à revenu positif, pour un revenu maximal de 2 : l'ignorance de la dimension paiement coûtait **100 %** du revenu, pas une marge.
- Le mécanisme extracteur est non-déviation locale **par indifférence** (sincère `u = 0`, déviation `u = 0`) et IR serré ; sur `(0, 0)`, `(0, 1)` et `(1, 0)`, aucun candidat à paiement uniforme ≥ 1 n'est admissible — le témoin est vrai sur ces instances, et le carnet montre sa **bordure** sur `(1, 1)`, où un mécanisme à paiement non nul survit au point-prior.


## Vérifiez votre compréhension

1. Le designer de `GameTheory-16b` atteint le bien-être optimal `J = 2` sur `(1, 1)` mais encaisse un revenu nul. Le mécanisme est-il irrationnel ?

   <details><summary>Réponse</summary>

   Il maximise le bien-être demandé, mais ce critère seul ne garantit pas l’admissibilité : la vérification explicite montre que sa table viole la non-déviation locale. Le revenu nul et le bien-être maximal restent des mesures correctes. Le carnet mesure l'écart entre l'espace que le générateur visite (16 tables d'issues) et l'espace complet (4096 candidats) : sur les 4096, il existe 47 mécanismes admissibles, dont 34 à bien-être optimal et **29** à revenu positif. Le bien-être est atteint dans les deux cas ; le revenu, non. L'ignorance de la dimension paiement n'a pas coûté une marge, elle a coûté **100 %** du revenu — d'où la méthode : mesurer l'écart sur **tous** les critères du problème, pas seulement celui qu'on optimise.

   </details>

2. Le mécanisme extracteur est non-déviation locale « par indifférence » : l'agent sincère obtient `u = 0` et l'agent qui dévie obtient aussi `u = 0`. Un agent indifférent est-il bien incitivé ?

   <details><summary>Réponse</summary>

   Oui — l'indifférence est le cas limite le plus exigeant de l'incitation. non-déviation locale demande que le report sincère soit une **stratégie dominante**, c'est-à-dire optimal au sens large ; l'égalité stricte l'est aussi, à condition que **toute** autre déviation donne une utilité au plus égale. Ici l'écart est nul exactement parce que le paiement extrait toute la valeur : l'agent de type 1 paie 1 et reçoit `1 · 1`, l'agent de type 0 reçoit 0 et paie 0. Il ne peut ni gagner ni perdre en mentant — et le carnet le vérifie par énumération, pas par l'affirmation.

   </details>

3. Le témoin d'impossibilité de `GameTheory-16b` est annoncé « vrai » sur `(0, 0)`, `(0, 1)` et `(1, 0)`, mais sur `(1, 1)` un mécanisme à paiement non nul survit. Une impossibilité « vraie » qui a une bordure, qu'est-ce que cela veut dire ?

   <details><summary>Réponse</summary>

   Que la preuve et le vérificateur **raisonnent dans deux sémantiques différentes**, et que le carnet le nomme au lieu de le cacher. La preuve raisonne **uniforme sur les types et reports** (le point-prior est `(1, 1)` : `θ = 0` force `p = 0` partout) ; le vérificateur raisonne **point-prior** (le profil observé est `(1, 1)`, où `θ = 1` pour les deux agents). Sur les instances où un agent de type 0 existe, les deux lectures s'accordent : le témoin est vrai. Sur `(1, 1)`, elles divergent — 29 mécanismes à revenu positif d'un côté, `paiements ≡ 0` en théorème de l'autre. Une impossibilité sans contre-témoin surnomme sa contrainte ; avec contre-témoin, elle en montre la **bordure**.

   </details>

4. La forme close du revenu maximal (`Σ θᵢ`) est-elle un théorème général d'AMD ?

   <details><summary>Réponse</summary>

   Non — le carnet le dit explicitement : c'est une **instance**, pas un théorème. L'extraction complète est la propriété du **prior ponctuel** : sur `(1, 1)`, `Σ θᵢ = 2` est atteint, et le mécanisme est non-déviation locale par indifférence. Sortir du prior ponctuel (des distributions sur les types) est justement la « vraie AMD » que le carnet nomme en perspective : les 4096 candidats deviennent alors des familles indexées par les distributions. Les trois perspectives ouvertes sont le coût d'issue nul (le bien public de Green-Laffont), le non-déviation locale strict et cette sortie du prior ponctuel.

   </details>


## Pour aller plus loin

**Référence pédagogique primaire.** Le calcul porté et relu est le [source Python versionné à a8566836](https://github.com/jsboige/CoursIA/blob/a8566836bb4246cfedbf6923f63fb0b2ab37d368/MyIA.AI.Notebooks/GameTheory/GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb), avec ses énumérations et ses vérificateurs explicites. Les expériences du présent carnet vérifient ce modèle local, pas les théorèmes d’une publication externe.

**Contexte historique conservé du source.** Les notices ci-dessous étaient présentes dans ce source et situent le thème ; aucune nouvelle lecture de copie archivée ni vérification Crossref de première main n’est revendiquée par ce port. L’IR robuste de ce jouet ne reproduit pas l’IR ex-interim usuelle.

- Green, J., & Laffont, J.-J. (1977). Characterization of satisfactory mechanisms for the revelation of preferences for public goods. *Econometrica*, 45(2), 427. https://doi.org/10.2307/1911219
- Rochet, J.-C. (1987). A necessary and sufficient condition for rationalizability in a quasi-linear context. *Journal of Mathematical Economics*, 16(2), 191-200. https://doi.org/10.1016/0304-4068(87)90007-3
- Saijo, T. (1988). Strategy space reduction in Maskin's theorem: sufficient conditions for Nash implementation. *Econometrica*, 56(3), 693. https://doi.org/10.2307/1911706
- Myerson, R. B., & Satterthwaite, M. A. (1983). Efficient mechanisms for bilateral trading. *Journal of Economic Theory*, 29(2), 265-281. https://doi.org/10.1016/0022-0531(83)90048-0

**Carnet suivant.** [GameTheory-16d — Joueurs LLM (Othman-Sandholm)](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) -- du designer qui optimise dans un espace borné aux **joueurs** qui ne sont pas scriptés : le gel pré-enregistré et les contrôles scriptés y jouent le rôle que le vérificateur séparé joue ici.

**Carnets voisins.**

- [GameTheory-16b — Automated Mechanism Design](GameTheory-16b-Automated-Mechanism-Design-Python.ipynb) -- l'organe repris tel quel, et le générateur dont le carnet mesure l'effondrement.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : principe de révélation, VCG, et les limites que le théorème d'impossibilité pose à toute conception de mécanisme.
- [GameTheory-16k — Théorie des enchères (Myerson)](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb) -- le même Myerson, côté vendeur : la réserve qui retire la rente d'information.
